In [ ]:
!pip install -q x-transformers
!pip install -q flash-attn --no-build-isolation

import torch
import torch.nn as nn
import torch.optim as optim
import math
import os
import hashlib
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import RobertaTokenizerFast, get_cosine_schedule_with_warmup, DataCollatorForLanguageModeling
from datasets import load_dataset
from x_transformers import TransformerWrapper, Encoder

# ==========================================
# 1. CONFIGURATION
# ==========================================
HF_ID = "prism-lab/wikitext-103-prism-32k-seq4k"
EXPERIMENT_NAME = "BASELINE_Pure_XTransformers"

VOCAB_SIZE = 32768
SEQ_LEN = 4096
BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS = 40
LR = 1e-3
D_MODEL = 512
DEPTH = 5
HEADS = 8
DROPOUT = 0.1
WEIGHT_DECAY = 0.01

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
def print_detailed_param_count(model):
    """
    Detailed parameter breakdown for x_transformers models.
    """
    total_params = 0
    categories = {
        "Embeddings": 0,
        "Attention": 0,
        "FeedForward": 0,
        "Norms": 0,
        "Head/Other": 0
    }

    # Track distinct tensors to handle tied weights correctly
    seen_pointers = set()

    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check for shared weights (tied embeddings)
            if param.data_ptr() in seen_pointers:
                continue
            seen_pointers.add(param.data_ptr())

            num_params = param.numel()
            total_params += num_params

            # Logic for x_transformers naming conventions
            if "token_emb" in name or "pos_emb" in name:
                categories["Embeddings"] += num_params
            elif "attn" in name or "to_q" in name or "to_k" in name or "to_v" in name:
                categories["Attention"] += num_params
            elif "ff" in name:
                categories["FeedForward"] += num_params
            elif "norm" in name:
                categories["Norms"] += num_params
            else:
                categories["Head/Other"] += num_params

    print(f"\n{'='*60}")
    print(f"{'COMPONENT':<25} | {'PARAMS':<12} | {'%':<6}")
    print(f"{'-'*60}")

    for cat, count in categories.items():
        if count > 0:
            percentage = (count / total_params) * 100
            print(f"{cat:<25} | {count:12,} | {percentage:5.1f}%")

    print(f"{'='*60}")
    print(f"{'TOTAL':<25} | {total_params:12,} | 100.0%")
    print(f"{'='*60}\n")

    return total_params

# ==========================================
# 2. TRAINING ROUTINE
# ==========================================
def run_baseline_training():
    from google.colab import drive
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    # Setup Logging
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = hashlib.md5(timestamp.encode()).hexdigest()[:8]
    SAVE_DIR = os.path.join("/content/drive/My Drive/PRISM_Experiments", f"{EXPERIMENT_NAME}_{timestamp}_{run_id}")
    os.makedirs(SAVE_DIR, exist_ok=True)
    writer = SummaryWriter(log_dir=SAVE_DIR)

    # Data
    print("⬇️ Loading Data...")
    tokenizer = RobertaTokenizerFast.from_pretrained(HF_ID)
    dataset = load_dataset(HF_ID)
    pad_id = tokenizer.pad_token_id # We need this for the mask

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

    train_loader = DataLoader(dataset["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator, num_workers=2, pin_memory=True, persistent_workers=True)
    valid_loader = DataLoader(dataset["validation"], batch_size=BATCH_SIZE, collate_fn=data_collator, num_workers=2, pin_memory=True)
    test_loader = DataLoader(dataset["test"], batch_size=BATCH_SIZE, collate_fn=data_collator, num_workers=2, pin_memory=True)

    # ======================================================
    # 3. MODEL: DIRECTLY USING THE LIBRARY
    # ======================================================
    print("\n⚡ INITIALIZING x_transformers LIBRARY MODEL...")

    model = TransformerWrapper(
        num_tokens=VOCAB_SIZE,
        max_seq_len=SEQ_LEN,
        use_abs_pos_emb=False,    # FALSE because we use RoPE
        tie_embedding=True,       # Match PRISM (if PRISM tied embeddings)
        attn_layers=Encoder(
            dim=D_MODEL,
            depth=DEPTH,
            heads=HEADS,
            layer_dropout=DROPOUT,
            attn_dropout=DROPOUT,
            ff_dropout=DROPOUT,
            rotary_pos_emb=True,  # Match PRISM Refiner
            attn_flash=True,      # Match PRISM Refiner
            use_scalenorm=False   # <--- CHANGE THIS TO FALSE (Or remove it)
        )
    ).to(DEVICE)
    print(model)
    print_detailed_param_count(model)
    # Simple Parameter Count
    print(f"📊 Parameters: {sum(p.numel() for p in model.parameters()):,}")

    # ======================================================

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()

    print(f"\n🚀 STARTING TRAINING (Ep 1 to {EPOCHS})")

    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS}", dynamic_ncols=True)

        for step, batch in enumerate(pbar):
            x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)

            # --- CRITICAL CHANGE: PASS MASK MANUALLY ---
            # x_transformers requires 'mask' to know what is padding
            mask = (x != pad_id)

            with torch.cuda.amp.autocast():
                # We pass the mask directly here
                outputs = model(x, mask=mask)
                loss = criterion(outputs.view(-1, VOCAB_SIZE), y.view(-1)) / GRAD_ACCUM

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                loss_val = loss.item() * GRAD_ACCUM
                pbar.set_postfix({'loss': f"{loss_val:.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.2e}"})
                writer.add_scalar('Train/Loss', loss_val, global_step)

        # Validation
        model.eval()
        val_loss = 0
        total_batches = 0
        with torch.no_grad():
            for batch in valid_loader:
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                mask = (x != pad_id)

                with torch.cuda.amp.autocast():
                    outputs = model(x, mask=mask)
                    val_loss += criterion(outputs.view(-1, VOCAB_SIZE), y.view(-1)).item()
                total_batches += 1

        avg_val_loss = val_loss / total_batches
        ppl = math.exp(avg_val_loss) if avg_val_loss < 100 else float('inf')
        print(f"✨ Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | PPL: {ppl:.2f}")
        writer.add_scalar('Val/PPL', ppl, epoch+1)

        # Save Best
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best.pt"))
            print("   🏆 New Best Model Saved!")

        # Save Last State (Optimizer, Scheduler, Scaler)
        state = {
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'best_loss': best_val_loss
        }
        torch.save(state, os.path.join(SAVE_DIR, "last.pt"))

    # Test
    print(f"\n🧪 Testing Best Model...")
    if os.path.exists(os.path.join(SAVE_DIR, "best.pt")):
        model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best.pt")))

    model.eval()
    test_loss = 0
    total_batches = 0
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
            mask = (x != pad_id)
            with torch.cuda.amp.autocast():
                outputs = model(x, mask=mask)
                test_loss += criterion(outputs.view(-1, VOCAB_SIZE), y.view(-1)).item()
            total_batches += 1

    final_ppl = math.exp(test_loss / total_batches)
    print(f"🏆 FINAL BASELINE PPL: {final_ppl:.2f}")

    writer.close()
    return model

if __name__ == "__main__":
    run_baseline_training()

In [ ]:
!pip freeze

In [ ]:
import sys
import torch
import platform
import os

# Define output file
log_file = "environment_snapshot.txt"

print(f"📝 Generating {log_file}...")

with open(log_file, "w") as f:
    # 1. PYTHON & SYSTEM INFO
    f.write("="*40 + "\n")
    f.write("SYSTEM INFORMATION\n")
    f.write("="*40 + "\n")
    f.write(f"Python Version: {sys.version}\n")
    f.write(f"Platform:       {platform.platform()}\n")
    f.write(f"Architecture:   {platform.machine()}\n")
    f.write(f"Processor:      {platform.processor()}\n")

    # 2. GPU & CUDA INFO
    f.write("\n" + "="*40 + "\n")
    f.write("GPU / CUDA INFORMATION\n")
    f.write("="*40 + "\n")
    f.write(f"PyTorch Version: {torch.__version__}\n")
    f.write(f"CUDA Available:  {torch.cuda.is_available()}\n")

    if torch.cuda.is_available():
        f.write(f"CUDA Version:    {torch.version.cuda}\n")
        f.write(f"CUDNN Version:   {torch.backends.cudnn.version()}\n")
        f.write(f"Device Name:     {torch.cuda.get_device_name(0)}\n")
        f.write(f"Device Count:    {torch.cuda.device_count()}\n")
    else:
        f.write("NO GPU DETECTED\n")

    # 3. COLAB SPECIFICS (If applicable)
    f.write("\n" + "="*40 + "\n")
    f.write("ENV VARIABLES (FILTERED)\n")
    f.write("="*40 + "\n")
    # Capturing useful Colab/Jupyter env vars without exposing secrets
    keys_to_log = ['COLAB_GPU', 'CUDA_VERSION', 'TBE_RUNTIME_ADDR']
    for k, v in os.environ.items():
        if any(x in k for x in ['COLAB', 'CUDA', 'LD_LIBRARY_PATH']):
            f.write(f"{k}: {v}\n")

    # 4. INSTALLED PACKAGES (Pip Freeze)
    f.write("\n" + "="*40 + "\n")
    f.write("PIP FREEZE (FULL LIBRARY LIST)\n")
    f.write("="*40 + "\n")

# Append pip freeze output directly to the file
os.system(f"pip freeze >> {log_file}")

print(f"✅ Log saved to {log_file}")
print("   (Check the Files tab on the left to download it)")

# Optional: Print head of file to verify
!head -n 20 environment_snapshot.txt